In [0]:
import os
import sys

from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "lakemeter_catalog")
dbutils.widgets.text("schema", "ecommerce_dev")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

sys.dont_write_bytecode = True
lib_path = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src", "pipeline"))
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from ecommerce_lib import transforms as t
from ecommerce_lib.schemas import LAYER_COLUMNS
from ecommerce_lib.synthetic import EXPECTED_COUNTS

results = []


def check(layer, name, passed, detail=""):
    results.append((layer, name, "PASS" if passed else "FAIL", str(detail)))


def tbl(name):
    return spark.read.table(f"`{catalog}`.`{schema}`.`{name}`")


print(f"Integration testing {catalog}.{schema}")

In [0]:
for name, expected_cols in LAYER_COLUMNS.items():
    layer = name.split("_")[0]
    try:
        df = tbl(name)
    except Exception as e:
        check(layer, f"{name} exists", False, e)
        continue
    check(layer, f"{name} columns", df.columns == expected_cols, df.columns)
    if name in EXPECTED_COUNTS:
        n = df.count()
        check(layer, f"{name} row count", n == EXPECTED_COUNTS[name], f"{n} (expected {EXPECTED_COUNTS[name]})")

In [0]:
for name, rules, key in [
    ("silver_customers", t.CUSTOMER_RULES, "customer_id"),
    ("silver_products", t.PRODUCT_RULES, "product_id"),
    ("silver_orders", t.ORDER_RULES, "order_id"),
]:
    df = tbl(name)
    total = df.count()
    check("silver", f"{name} passes all DQ rules", t.apply_rules(df, rules).count() == total)
    check("silver", f"{name} unique {key}", df.select(key).distinct().count() == total)

# Rows dropped between bronze and silver are exactly the injected bad rows.
check("silver", "bad customer C051 dropped", tbl("silver_customers").filter("customer_id = 'C051'").count() == 0)
check("silver", "bad product P021 dropped", tbl("silver_products").filter("product_id = 'P021'").count() == 0)
check("silver", "bad orders O0301-O0303 dropped",
      tbl("silver_orders").filter(F.col("order_id").isin("O0301", "O0302", "O0303")).count() == 0)
check("silver", "silver_customers emails lower-cased",
      tbl("silver_customers").filter("email <> lower(trim(email))").count() == 0)

In [0]:
def total(df, col):
    return df.agg(F.sum(col)).first()[0]

revenue_orders = tbl("silver_orders").filter(F.col("status").isin(*t.REVENUE_STATUSES))
expected_revenue = total(revenue_orders, "line_amount")

daily_rev = total(tbl("gold_daily_revenue"), "revenue")
category_rev = total(tbl("gold_category_sales"), "revenue")
clv_rev = total(tbl("gold_customer_lifetime_value"), "lifetime_revenue")

check("gold", "daily revenue = silver revenue", daily_rev == expected_revenue, f"{daily_rev} vs {expected_revenue}")
check("gold", "category revenue = silver revenue", category_rev == expected_revenue, f"{category_rev} vs {expected_revenue}")
check("gold", "CLV revenue = silver revenue", clv_rev == expected_revenue, f"{clv_rev} vs {expected_revenue}")

n_days = revenue_orders.select("order_date").distinct().count()
n_daily = tbl("gold_daily_revenue").count()
check("gold", "one gold_daily_revenue row per order date", n_daily == n_days, f"{n_daily} vs {n_days}")
check("gold", "daily order_count = revenue orders",
      total(tbl("gold_daily_revenue"), "order_count") == revenue_orders.count())
check("gold", "no negative revenue",
      tbl("gold_customer_lifetime_value").filter("lifetime_revenue < 0").count() == 0)

In [0]:
report = spark.createDataFrame(results, "layer STRING, check STRING, status STRING, detail STRING")
display(report)

failed = [r for r in results if r[2] == "FAIL"]
print(f"{len(results) - len(failed)}/{len(results)} integration checks passed")
assert not failed, "Integration tests failed:\n" + "\n".join(f"  [{l}] {c}: {d}" for l, c, _, d in failed)